# Custom Video Anomaly Detection using MULDE

This notebook runs the **MULDE (Multiscale Log-Density Estimation)** pipeline on a custom `.mp4` video. It mounts Google Drive, pulls the latest runner script from your repository, installs the required video processing and model packages, runs the inference, and displays the resulting anomaly score chart.

## Step 1: Mount Google Drive
Run the cell below to authenticate and mount your Google Drive to access model checkpoints and save experiments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies
Install Hiera, timm, decord, and other necessary packages required for video loading and model execution.

In [ ]:
!pip install -q timm decord opencv-python-headless pandas tqdm matplotlib joblib scikit-learn
!pip install -q git+https://github.com/facebookresearch/hiera.git@b12b842542ee5c757fcfec8c41f6b56fcbe89b65

## Step 3: Clone or Update the Fusion Repository
Clone your repository to get the `run_mulde_on_custom_video.py` script, or pull updates if already cloned.

In [ ]:
import os
REPO_DIR = "/content/Fusion"
if not os.path.exists(REPO_DIR):
  !git clone https://github.com/Hadi6618/Fusion.git {REPO_DIR}
else:
  %cd {REPO_DIR}
  !git pull

%cd {REPO_DIR}

## Step 4: Configure and Run Anomaly Detection
Set the paths to your video, checkpoint, GMM model, and output directory. Then execute the MULDE pipeline.

In [ ]:
# Define your paths
VIDEO_PATH = "/content/name_of_video.mp4"
CHECKPOINT_PATH = "/content/drive/My Drive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/checkpoints/mulde_final.pt"
STATS_PATH = "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train_feature_stats.npz"
GMM_PATH = "/content/drive/My Drive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/gmm_components_5.joblib"
OUTPUT_DIR = "/content/drive/My Drive/Experiments"

# Optional: Set explosion frame ranges if you want to highlight/shade them on the chart (e.g. "120-180")
# Set to None if you do not know the ranges yet.
SHADING_RANGES = None  

# Build the execution command
cmd = f"python run_mulde_on_custom_video.py \
  --video '{VIDEO_PATH}' \
  --checkpoint '{CHECKPOINT_PATH}' \
  --stats '{STATS_PATH}' \
  --gmm '{GMM_PATH}' \
  --output_dir '{OUTPUT_DIR}' \
  --smooth_sigma 15.0"

if SHADING_RANGES:
  cmd += f" --shading '{SHADING_RANGES}'"

# Run the script
!{cmd}

## Step 5: Visualize the Anomaly Chart
Display the generated log-likelihood curve directly in the notebook.

In [ ]:
from IPython.display import Image, display
import os
from pathlib import Path

video_name = Path(VIDEO_PATH).stem
chart_file = os.path.join(OUTPUT_DIR, f"{video_name}_anomaly_profile.png")

if os.path.exists(chart_file):
  display(Image(filename=chart_file))
else:
  print(f"Chart not found at: {chart_file}. Check output logs above for errors.")

### Note on Shading Anomaly Windows
Once you view the chart and note where the anomaly score dips (representing the explosion scene), you can update the `SHADING_RANGES` variable in **Step 4** (e.g., `SHADING_RANGES = "120-180"`) and re-run the cells to get a chart with the explosion regions highlighted in pink.